## Setup and Data Loading

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import matplotlib
matplotlib.use('Agg')          # use non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from pathlib import Path

# output directories
STATIC_DIR = Path('../outputs/visualizations/static')
INTERACTIVE_DIR = Path('../outputs/visualizations/interactive')
STATIC_DIR.mkdir(parents=True, exist_ok=True)
INTERACTIVE_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded successfully')
print(f'matplotlib {matplotlib.__version__}, seaborn {sns.__version__}, plotly {plotly.__version__}')

Libraries loaded successfully
matplotlib 3.9.4, seaborn 0.13.2, plotly 6.7.0


In [2]:
# Load the cleaned articles dataset produced in Lab 9
# Works whether CWD is notebooks/ or the project root (e.g. after the generator cell runs)
_from_notebooks = Path('../data/processed/cleaned/articles_clean.csv')
_from_root      = Path('data/processed/cleaned/articles_clean.csv')
DATA_PATH = _from_notebooks if _from_notebooks.exists() else _from_root

df = pd.read_csv(DATA_PATH)
print(f'Loaded from: {DATA_PATH.resolve()}')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Loaded from: /Users/hamza/Desktop/Health-Wellness-Information-Pipeline/data/processed/cleaned/articles_clean.csv
Shape: (297, 23)
Columns: ['source_name', 'title', 'author', 'description', 'publishedAt', 'content', 'text', 'ID', 'Title', 'Source', 'Author', 'Published Date', 'Word Count', 'Category', 'Nutrition', 'Pharmacology', 'Mental Health', 'Total', 'raw_text', 'processed_text', 'published_date', 'image_url', 'published_year']


,source_name,title,author,description,publishedAt,content,text,ID,Title,Source,...,Category,Nutrition,Pharmacology,Mental Health,Total,raw_text,processed_text,published_date,image_url,published_year
0,Science Daily,Scientists discover diet that tricks the body ...,NaN,Researchers found that cutting two amino acids...,2026-02-27 18:05:43+00:00,"Shivering in the cold is uncomfortable, but it...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026
1,Naturalnews.com,Nature’s powerhouses: The top 8 healthiest ber...,Belle Carter,"Berries are rich in vitamin C, fiber and polyp...",2026-03-12 06:00:00+00:00,"<ul><li>Berries are rich in vitamin C, fiber a...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026
2,Eatingwell.com,6 Things You Should Do After 5 P.M. to Support...,Cheyenne Buckingham,Aging is a privilege—support your health in la...,2026-03-13 21:30:00+00:00,"Reviewed by Dietitian Sarah Pflugradt, Ph.D., ...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026


In [3]:
# Quick overview of key columns
df['content_length'] = df['content'].str.len().fillna(0).astype(int)
print(f"Year range: {df['published_year'].min()} – {df['published_year'].max()}")
print(f"Unique sources: {df['source_name'].nunique()}")
print(f"Unique authors: {df['author'].nunique()}")
print()
df[['published_year', 'content_length']].describe().round(2)

Year range: 2026 – 2026
Unique sources: 81
Unique authors: 186



,published_year,content_length
count,297.0,297.00
mean,2026.0,212.84
std,0.0,10.07
min,2026.0,110.00
25%,2026.0,213.00
50%,2026.0,214.00
75%,2026.0,214.00
max,2026.0,272.00


---
## matplotlib Static Plots

### Visualization Principles

| Question | Chart type |
|----------|------------|
| Which movies earned the most? | Horizontal **bar** chart |
| How has the average rating changed over time? | **Line** chart |
| Is there a relationship between budget and revenue? | **Scatter** plot |
| What does the rating distribution look like? | **Histogram** + KDE |
| How do ratings vary across genres? | **Box** plot |
| What correlates with what? | **Heatmap** |

In [4]:
# Apply seaborn theme globally (Part 3 of the lecture – Styling)
sns.set_theme(style='whitegrid')
sns.set_context('notebook')
sns.set_palette('viridis')

### Bar Chart – Top 10 Movies by Revenue

We use `ax.barh()` (horizontal bar) because the movie titles are long strings.  
The **object-oriented API** (`fig, ax = plt.subplots()`) is preferred per the lecture material.

In [5]:
top10 = (df['source_name'].dropna()
           .value_counts()
           .head(10)
           .sort_values())

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('viridis', 10)
bars = ax.barh(top10.index, top10.values, color=colors)

for bar in bars:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(int(bar.get_width())), va='center', fontsize=9)

ax.set_xlabel('Number of Articles', fontsize=11)
ax.set_title('Top 10 Sources by Article Count', fontsize=14, fontweight='bold')
ax.set_xlim(0, top10.max() * 1.15)
fig.tight_layout()

out_png = STATIC_DIR / 'top_sources_by_count.png'
out_pdf = STATIC_DIR / 'top_sources_by_count.pdf'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
fig.savefig(out_pdf, bbox_inches='tight')
plt.show()
print(f'Saved → {out_png}  and  {out_pdf}')

Saved → ../outputs/visualizations/static/top_sources_by_count.png  and  ../outputs/visualizations/static/top_sources_by_count.pdf


/var/folders/g2/l6z680ks5g73j427gfgb5gy80000gn/T/ipykernel_56110/2242617597.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Line Chart – Average Rating Over Years

Dual-axis line + bar chart: bars show the number of movies per year, the line shows the average vote rating.

In [6]:
yearly = (df.groupby('published_year')
            .agg(article_count=('title', 'count'),
                 avg_content_length=('content_length', 'mean'))
            .reset_index()
            .query('published_year >= 2000'))

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(yearly['published_year'], yearly['article_count'], color='#a8d5e2', alpha=0.5, label='Article Count')
ax1.set_ylabel('Number of Articles', color='#a8d5e2', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#a8d5e2')

ax2 = ax1.twinx()
ax2.plot(yearly['published_year'], yearly['avg_content_length'],
         color='#1a6faf', linewidth=2.5, marker='o', markersize=5, label='Avg Content Length')
ax2.set_ylabel('Avg Content Length (chars)', color='#1a6faf', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#1a6faf')

ax1.set_xlabel('Publication Year', fontsize=11)
ax1.set_title('Health Article Volume and Avg Content Length Over the Years', fontsize=14, fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
fig.tight_layout()

fig.savefig(STATIC_DIR / 'articles_over_years.png', dpi=300, bbox_inches='tight')
fig.savefig(STATIC_DIR / 'articles_over_years.pdf', bbox_inches='tight')
plt.show()
print('Saved → articles_over_years  (PNG + PDF)')

Saved → articles_over_years  (PNG + PDF)


/var/folders/g2/l6z680ks5g73j427gfgb5gy80000gn/T/ipykernel_56110/176555522.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Scatter Plot – Budget vs Revenue

In [7]:
source_stats = (df.groupby('source_name')
                  .agg(article_count=('title', 'count'),
                       avg_length=('content_length', 'mean'))
                  .reset_index())

fig, ax = plt.subplots(figsize=(11, 7))
scatter = ax.scatter(source_stats['article_count'],
                     source_stats['avg_length'] / 1000,
                     c=source_stats['article_count'],
                     cmap='viridis', alpha=0.8, s=80, edgecolors='white')

for _, row in source_stats.iterrows():
    ax.annotate(row['source_name'],
                (row['article_count'], row['avg_length'] / 1000),
                textcoords='offset points', xytext=(5, 5), fontsize=7, alpha=0.85)

plt.colorbar(scatter, ax=ax, label='Article Count')
ax.set_xlabel('Number of Articles', fontsize=11)
ax.set_ylabel('Avg Content Length (thousand chars)', fontsize=11)
ax.set_title('Source Productivity: Article Count vs Avg Content Length', fontsize=14, fontweight='bold')
fig.tight_layout()

fig.savefig(STATIC_DIR / 'source_count_vs_length.png', dpi=300, bbox_inches='tight')
fig.savefig(STATIC_DIR / 'source_count_vs_length.pdf', bbox_inches='tight')
plt.show()
print('Saved → source_count_vs_length  (PNG + PDF)')

Saved → source_count_vs_length  (PNG + PDF)


/var/folders/g2/l6z680ks5g73j427gfgb5gy80000gn/T/ipykernel_56110/1310168809.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## seaborn Statistical Plots

seaborn is built on matplotlib. `set_theme()`, `set_context()`, `set_palette()` provide consistent styling.

### Histogram + KDE – Rating Distribution

In [8]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df['content_length'].dropna() / 1000, bins=25, kde=True,
             color='#1a6faf', edgecolor='white', ax=ax)
ax.axvline(df['content_length'].mean() / 1000, color='firebrick', linestyle='--', linewidth=1.8,
           label=f"Mean = {df['content_length'].mean() / 1000:.1f}K chars")
ax.axvline(df['content_length'].median() / 1000, color='darkorange', linestyle='-.', linewidth=1.8,
           label=f"Median = {df['content_length'].median() / 1000:.1f}K chars")
ax.set_xlabel('Content Length (thousand chars)', fontsize=11)
ax.set_ylabel('Number of Articles', fontsize=11)
ax.set_title('Distribution of Article Content Lengths', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
fig.tight_layout()

fig.savefig(STATIC_DIR / 'content_length_distribution.png', dpi=300, bbox_inches='tight')
fig.savefig(STATIC_DIR / 'content_length_distribution.pdf', bbox_inches='tight')
plt.show()
print('Saved → content_length_distribution  (PNG + PDF)')

Saved → content_length_distribution  (PNG + PDF)


/var/folders/g2/l6z680ks5g73j427gfgb5gy80000gn/T/ipykernel_56110/2095617414.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Box Plot – Ratings by Genre

In [9]:
top_sources = df['source_name'].value_counts().head(10).index.tolist()
data = df[df['source_name'].isin(top_sources)]
order = (data.groupby('source_name')['content_length']
             .median().sort_values(ascending=False).index.tolist())

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=data, x='source_name', y='content_length',
            order=order, hue='source_name', palette='viridis', legend=False, ax=ax)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.set_xlabel('Source', fontsize=11)
ax.set_ylabel('Content Length (chars)', fontsize=11)
ax.set_title('Content Length Distribution by Top 10 Sources', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
fig.tight_layout()

fig.savefig(STATIC_DIR / 'content_length_by_source.png', dpi=300, bbox_inches='tight')
fig.savefig(STATIC_DIR / 'content_length_by_source.pdf', bbox_inches='tight')
plt.show()
print('Saved → content_length_by_source  (PNG + PDF)')

Saved → content_length_by_source  (PNG + PDF)


/var/folders/g2/l6z680ks5g73j427gfgb5gy80000gn/T/ipykernel_56110/3762760648.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Correlation Heatmap – seaborn

In [10]:
top_sources = df['source_name'].value_counts().head(12).index.tolist()
pivot = (df[df['source_name'].isin(top_sources)]
           .groupby(['source_name', 'published_year'])
           .size()
           .unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns) * 0.9), 7))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.4, ax=ax, annot_kws={'size': 9})
ax.set_title('Article Count by Source and Publication Year',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Publication Year', fontsize=11)
ax.set_ylabel('Source', fontsize=11)
ax.tick_params(axis='y', rotation=0)
fig.tight_layout()

fig.savefig(STATIC_DIR / 'source_year_heatmap.png', dpi=300, bbox_inches='tight')
fig.savefig(STATIC_DIR / 'source_year_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Saved → source_year_heatmap  (PNG + PDF)')

Saved → source_year_heatmap  (PNG + PDF)


/var/folders/g2/l6z680ks5g73j427gfgb5gy80000gn/T/ipykernel_56110/4060903718.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Subplot Layout

In [11]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Health & Wellness Articles Dashboard', fontsize=16, fontweight='bold', y=1.01)

# Panel A – top 10 sources by article count
counts_src = df['source_name'].dropna().value_counts().head(10).sort_values()
axes[0,0].barh(counts_src.index, counts_src.values, color=sns.color_palette('viridis', 10))
axes[0,0].set_title('Top 10 Sources by Article Count', fontsize=11, fontweight='bold')
axes[0,0].set_xlabel('Number of Articles')

# Panel B – content length distribution
sns.histplot(df['content_length'].dropna() / 1000, bins=20, kde=True, color='#1a6faf', ax=axes[0,1])
axes[0,1].set_title('Content Length Distribution', fontsize=11, fontweight='bold')
axes[0,1].set_xlabel('Content Length (thousand chars)')
axes[0,1].set_ylabel('Count')

# Panel C – article volume over years
yearly = (df.query('published_year >= 2000')
            .groupby('published_year').size()
            .reset_index(name='article_count'))
axes[1,0].bar(yearly['published_year'], yearly['article_count'],
              color=sns.color_palette('viridis', len(yearly)))
axes[1,0].set_title('Article Volume Over Years', fontsize=11, fontweight='bold')
axes[1,0].set_xlabel('Year')
axes[1,0].set_ylabel('Count')

# Panel D – top 8 authors
counts_auth = df['author'].dropna().value_counts().head(8)
axes[1,1].bar(counts_auth.index, counts_auth.values,
              color=sns.color_palette('viridis', len(counts_auth)))
axes[1,1].set_title('Top 8 Authors by Article Count', fontsize=11, fontweight='bold')
axes[1,1].set_xlabel('Author')
axes[1,1].set_ylabel('Count')
axes[1,1].tick_params(axis='x', rotation=30)

fig.tight_layout()
fig.savefig(STATIC_DIR / 'dashboard_subplots.png', dpi=300, bbox_inches='tight')
fig.savefig(STATIC_DIR / 'dashboard_subplots.pdf', bbox_inches='tight')
plt.show()
print('Saved → dashboard_subplots  (PNG + PDF)')

Saved → dashboard_subplots  (PNG + PDF)


/var/folders/g2/l6z680ks5g73j427gfgb5gy80000gn/T/ipykernel_56110/428504356.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Plotly Express Interactive Charts

Plotly produces **interactive web charts** with hover tooltips, zoom, pan, and legend toggle — all without extra code.

### Interactive Scatter – Budget vs Revenue

In [12]:
data = df.copy()
data['author'] = data['author'].fillna('Unknown')

fig = px.scatter(
    data,
    x='published_year',
    y='content_length',
    color='source_name',
    hover_name='title',
    hover_data={
        'author': True,
        'source_name': True,
        'published_year': True,
        'content_length': True,
    },
    labels={
        'published_year': 'Publication Year',
        'content_length': 'Content Length (chars)',
        'source_name': 'Source',
    },
    title='Article Content Length Over Time – by Source',
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Vivid,
)
fig.update_layout(legend_title='Source', font=dict(family='Inter', size=13), height=580)

html_path = INTERACTIVE_DIR / 'content_length_scatter_interactive.html'
fig.write_html(str(html_path))
fig.show()
print(f'Saved → {html_path}')

Saved → ../outputs/visualizations/interactive/content_length_scatter_interactive.html


### Interactive Bar – Top 10 by Popularity

In [13]:
source_stats = (df.groupby('source_name')
                  .agg(article_count=('title', 'count'),
                       avg_content_length=('content_length', 'mean'),
                       unique_authors=('author', 'nunique'))
                  .reset_index()
                  .nlargest(10, 'article_count')
                  .sort_values('article_count', ascending=True))
source_stats['avg_content_length'] = source_stats['avg_content_length'].round(0).astype(int)

fig = px.bar(
    source_stats,
    x='article_count', y='source_name', orientation='h',
    color='avg_content_length',
    color_continuous_scale='Viridis',
    hover_name='source_name',
    hover_data={'article_count': True, 'avg_content_length': True, 'unique_authors': True},
    labels={'article_count': 'Number of Articles', 'source_name': 'Source',
            'avg_content_length': 'Avg Content Length (chars)'},
    title='Top 10 Sources by Article Count',
    template='plotly_white',
)
fig.update_layout(coloraxis_colorbar_title='Avg Length',
                  font=dict(family='Inter', size=13), height=500)

html_path = INTERACTIVE_DIR / 'top_sources_bar_interactive.html'
fig.write_html(str(html_path))
fig.show()
print(f'Saved → {html_path}')

Saved → ../outputs/visualizations/interactive/top_sources_bar_interactive.html


### Interactive Line – Movies per Year

In [14]:
yearly = (df.query('published_year >= 2000')
            .groupby('published_year')
            .agg(article_count=('title', 'count'),
                 avg_content_length=('content_length', 'mean'),
                 unique_authors=('author', 'nunique'),
                 unique_sources=('source_name', 'nunique'))
            .reset_index())
yearly['avg_content_length'] = yearly['avg_content_length'].round(0).astype(int)

fig = px.line(yearly, x='published_year', y='article_count', markers=True,
              hover_data={'avg_content_length': True, 'unique_authors': True, 'unique_sources': True},
              labels={'published_year': 'Year', 'article_count': 'Number of Articles'},
              title='Health Articles Published per Year',
              template='plotly_white')
fig.update_traces(line_color='#1a6faf', line_width=2.5, marker=dict(size=7))
fig.update_layout(font=dict(family='Inter', size=13), height=450)

html_path = INTERACTIVE_DIR / 'articles_per_year_line.html'
fig.write_html(str(html_path))
fig.show()
print(f'Saved → {html_path}')

Saved → ../outputs/visualizations/interactive/articles_per_year_line.html


### Interactive Box – Genre Ratings

In [15]:
top_sources = df['source_name'].value_counts().head(10).index.tolist()
data = df[df['source_name'].isin(top_sources)].copy()
data['author'] = data['author'].fillna('Unknown')
order = (data.groupby('source_name')['content_length']
              .median().sort_values(ascending=False).index.tolist())

fig = px.box(
    data, x='source_name', y='content_length',
    category_orders={'source_name': order},
    color='source_name',
    hover_name='title',
    hover_data={'author': True, 'published_year': True, 'content_length': True},
    labels={'source_name': 'Source', 'content_length': 'Content Length (chars)'},
    title='Content Length Distribution by Top 10 Sources',
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Vivid,
)
fig.update_layout(showlegend=False, font=dict(family='Inter', size=13), height=500)

html_path = INTERACTIVE_DIR / 'source_content_boxplot_interactive.html'
fig.write_html(str(html_path))
fig.show()
print(f'Saved → {html_path}')

Saved → ../outputs/visualizations/interactive/source_content_boxplot_interactive.html


---
### Multi-Chart Plotly Layout (2×2 Dashboard)

In [16]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Top 10 Sources by Article Count', 'Content Length Distribution',
                    'Articles per Year', 'Top 10 Authors by Article Count'),
    vertical_spacing=0.14, horizontal_spacing=0.10,
)

# 1 – top 10 sources horizontal bar
top_src = df['source_name'].dropna().value_counts().head(10).sort_values()
fig.add_trace(go.Bar(x=top_src.values, y=top_src.index, orientation='h',
                     marker_color='#1a6faf', name='Sources',
                     hovertemplate='%{y}<br>%{x} articles<extra></extra>'), row=1, col=1)

# 2 – content length histogram
fig.add_trace(go.Histogram(x=df['content_length'].dropna(), nbinsx=25,
                           marker_color='#2ca02c', name='Content Length',
                           hovertemplate='Length: %{x}<br>Count: %{y}<extra></extra>'), row=1, col=2)

# 3 – articles per year bar
yearly = (df.query('published_year >= 2000')
            .groupby('published_year').size()
            .reset_index(name='article_count'))
fig.add_trace(go.Bar(x=yearly['published_year'], y=yearly['article_count'],
                     marker_color='#ff7f0e', name='Articles per Year',
                     hovertemplate='Year: %{x}<br>Articles: %{y}<extra></extra>'), row=2, col=1)

# 4 – top 10 authors horizontal bar
top_auth = df['author'].dropna().value_counts().head(10).sort_values()
fig.add_trace(go.Bar(x=top_auth.values, y=top_auth.index, orientation='h',
                     marker_color='#9467bd', name='Authors',
                     hovertemplate='%{y}<br>%{x} articles<extra></extra>'), row=2, col=2)

fig.update_layout(title_text='Health & Wellness Articles Dashboard', title_font_size=18,
                  template='plotly_white', height=700, width=1100, showlegend=False,
                  font=dict(family='Inter', size=11))
fig.update_xaxes(title_text='Number of Articles', row=1, col=1)
fig.update_xaxes(title_text='Content Length (chars)', row=1, col=2)
fig.update_xaxes(title_text='Year', row=2, col=1)
fig.update_xaxes(title_text='Number of Articles', row=2, col=2)
fig.update_yaxes(title_text='Count', row=1, col=2)
fig.update_yaxes(title_text='Article Count', row=2, col=1)

html_path = INTERACTIVE_DIR / 'interactive_dashboard.html'
fig.write_html(str(html_path))
fig.show()
print(f'Saved → {html_path}')

Saved → ../outputs/visualizations/interactive/interactive_dashboard.html


---
## Automated Chart Generation

The `chart_generator.py` module wraps all chart functions into a single `generate_all()` call.  
Running the script from the project root generates all 13 charts automatically.

In [17]:
# Demonstrate the automated generator
from visualization.chart_generator import generate_all
results = generate_all()

print('\nStatic charts saved:')
for name, paths in results['static'].items():
    print(f'  {name}: {paths["png"]}')
print('\nInteractive charts saved:')
for name, path in results['interactive'].items():
    print(f'  {name}: {path}')


  Lab 12 – Data Visualization Generator

Dataset: 297 articles, 23 columns

── Static charts (matplotlib / seaborn) ──
  [static]  top_sources_by_count
            PNG → /Users/hamza/Desktop/Health-Wellness-Information-Pipeline/outputs/visualizations/static/top_sources_by_count.png
            PDF → /Users/hamza/Desktop/Health-Wellness-Information-Pipeline/outputs/visualizations/static/top_sources_by_count.pdf
  [static]  articles_over_years
            PNG → /Users/hamza/Desktop/Health-Wellness-Information-Pipeline/outputs/visualizations/static/articles_over_years.png
            PDF → /Users/hamza/Desktop/Health-Wellness-Information-Pipeline/outputs/visualizations/static/articles_over_years.pdf
  [static]  source_count_vs_length
            PNG → /Users/hamza/Desktop/Health-Wellness-Information-Pipeline/outputs/visualizations/static/source_count_vs_length.png
            PDF → /Users/hamza/Desktop/Health-Wellness-Information-Pipeline/outputs/visualizations/static/source_count_vs_len

---
## Document Visualization Choices

This section explains **why each chart type was chosen** for each analysis question.
For every chart, the reasoning covers:
- What question it answers
- Why that chart type matches the statistical nature of the data
- What encoding decisions were made (colour, size, axis scale)


### Bar Chart (Horizontal) – Top 10 Movies by Revenue

The question is a **ranking comparison across named categories** (film titles). Horizontal bars are preferred over vertical bars when category labels are long strings, because they read left-to-right naturally without rotation. The `viridis` palette encodes rank via luminance — darker at the bottom, brighter at the top. Inline value annotations (`$X.XXB`) eliminate the need to scan the x-axis for every bar.

---

### Dual-Axis Line + Bar – Average Rating Over Years

Two related but differently scaled time series are displayed together: **movie count** (a count variable, encoded as bars on the left y-axis) and **average rating** (a continuous 0–10 variable, encoded as a line on the right y-axis). A dual-axis chart is justified here because both series share a temporal x-axis and a reader naturally wants to correlate them. `ax.twinx()` creates the second axis without a second figure.

---

### Scatter Plot – Budget vs Revenue

The central financial question is whether higher budgets yield higher revenue. A **scatter plot** is the canonical chart for two continuous variables where we want to see correlation, outliers, and clusters simultaneously. Genre is encoded as **hue** (categorical colour) to reveal whether genre moderates the budget–revenue relationship. Vote average is encoded as **marker size** because it is a secondary dimension. The diagonal break-even line makes profitability immediately visible — points above the line are profitable.

---

### Histogram + KDE – Rating Distribution

The question is about the **shape of a single continuous variable's distribution** — whether it is normal, skewed, or bi-modal. A histogram reveals raw frequency per bin; the KDE overlay smooths bin-boundary artefacts and shows the underlying density curve. Mean and median vertical reference lines highlight central tendency and any skew at a glance.

---

### Box-and-Whisker – Rating by Genre

When comparing **distributions of a continuous variable across many groups**, box plots are more space-efficient than overlaid histograms. They show the median, interquartile range (IQR), and outliers in a single compact glyph. Genres are sorted by median rating descending so the ranking is immediately readable. `hue='primary_genre'` with `legend=False` is the correct seaborn ≥ 0.12 pattern for colouring by the same column used on the x-axis.

---

### Correlation Heatmap

A correlation matrix is an N×N grid of pairwise statistics. A **heatmap** is the standard encoding: the diverging `coolwarm` palette maps negative correlations to cool blue and positive correlations to warm red, with white at zero. Annotating each cell with the numeric value prevents misreading from colour alone. `square=True` ensures every cell has equal visual weight regardless of figure dimensions.

---

### Vertical Bar – Genre Count

The question is a **frequency distribution across a discrete categorical variable**. Bar length is the most perceptually accurate channel for magnitude comparisons. Vertical orientation is used here because genre names are short and the chart is wider than it is tall. Inline count labels above each bar eliminate the need to read off the y-axis.

---

### 2×2 Multi-Panel Dashboard

An executive summary must convey several different questions at a glance. A **2×2 subplot grid** combines the four most informative individual views into a single shareable figure. The layout follows a natural reading order: top-left (revenue ranking), top-right (rating distribution), bottom-left (genre composition), bottom-right (budget/revenue relationship). `fig.tight_layout()` prevents panel labels from overlapping.

---

### Interactive Scatter (Plotly Express) – Budget vs Revenue

The static scatter plot raises a follow-up question: *'Which specific film is that outlier?'* Interactive **hover tooltips** answer it without overcrowding the chart with text labels. The Plotly version adds ROI to the tooltip so an analyst can assess the profitability of individual films. The legend is also clickable — genres can be hidden or isolated.

---

### Interactive Line – Movies per Year

A **time-series line chart** is the correct choice when the reader wants to track a single metric's evolution over time. Interactivity lets users zoom into specific decades and hover to see exact counts, average ratings, and total revenue for any individual year.

---

### Interactive Box Plot – Genre Ratings (Plotly)

The interactive version of the boxplot adds per-movie hover so the reader can identify which specific title is an outlier within a genre. `category_orders` keeps genres sorted by median, matching the static version. The legend is hidden (`showlegend=False`) because genre is already encoded on the x-axis.

---

### Interactive 2×2 Dashboard (Plotly Graph Objects)

The Plotly dashboard uses `make_subplots()` with `go` traces instead of Plotly Express because Express does not support multi-panel layouts. Each panel is independently zoomable and hoverable. The colorbar in Panel 4 encodes vote_average as a continuous colour scale, adding a third dimension to the budget/revenue scatter without adding a legend.
